# 02 — Collecte du dataset (SQLite + snowball sampling)

Objectif : construire un pipeline qui collecte des milliers de matchs réels (compositions + résultats) en respectant le rate limiting, avec dédoublonnage et reprise possible après interruption. Ça alimentera la Factorization Machine de la phase 3.

**Partie 1 : le schéma SQLite et les fonctions de lecture/écriture.**

## Concept 1 — Schéma SQLite

Deux tables :
- `matches` : le JSON brut de chaque match, `match_id` en clé primaire (donc automatiquement unique — impossible d'insérer deux fois le même match, c'est notre dédoublonnage).
- `explored_players` : les `puuid` dont on a déjà récupéré la liste de matchs — pour ne pas interroger deux fois le même joueur pendant le snowball sampling.

`IF NOT EXISTS` rend la cellule sûre à ré-exécuter plusieurs fois sans erreur (utile pendant le développement).

In [ ]:
import sqlite3

conn = sqlite3.connect("../data/matches.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS matches (
    match_id TEXT PRIMARY KEY,
    region TEXT NOT NULL,
    raw_json TEXT NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS explored_players (
    puuid TEXT PRIMARY KEY
)
""")

conn.commit()
print("Tables prêtes.")

## Concept 2 — Fonctions de lecture/écriture

Quelques principes :
- **Requêtes paramétrées** (`?` dans le SQL, valeurs passées à part) plutôt que d'insérer les valeurs directement dans la chaîne SQL — même si ici les données viennent d'une API de confiance, c'est le réflexe à prendre pour ne jamais être vulnérable à une injection SQL.
- **`INSERT OR IGNORE`** : si la clé primaire existe déjà, SQLite ignore l'insertion silencieusement au lieu de lever une erreur — pratique pour notre dédoublonnage, pas besoin de vérifier "existe déjà ?" avant chaque insertion.
- **`json.dumps`** : une table SQL ne stocke pas nativement un dict Python — on le sérialise en texte JSON pour le stocker dans la colonne `raw_json`.

In [ ]:
import json


def match_already_collected(match_id):
    cursor.execute("SELECT match_id FROM matches WHERE match_id = ?", (match_id,))
    return cursor.fetchone() is not None


def save_match(match_id, region, match_data):
    cursor.execute(
        "INSERT OR IGNORE INTO matches (match_id, region, raw_json) VALUES (?, ?, ?)",
        (match_id, region, json.dumps(match_data)),
    )
    conn.commit()


def player_already_explored(puuid):
    cursor.execute("SELECT puuid FROM explored_players WHERE puuid = ?", (puuid,))
    return cursor.fetchone() is not None


def mark_player_explored(puuid):
    cursor.execute("INSERT OR IGNORE INTO explored_players (puuid) VALUES (?)", (puuid,))
    conn.commit()


print("Fonctions prêtes.")

## Petit test de sanité (sans appel API)

On vérifie que les fonctions marchent avec des données factices, avant de les brancher sur de vraies données Riot.

In [ ]:
fake_id = "TEST_1234"

print("Avant insertion :", match_already_collected(fake_id))  # attendu: False

save_match(fake_id, "euw1", {"hello": "world"})

print("Après insertion :", match_already_collected(fake_id))  # attendu: True

# on nettoie tout de suite, c'était juste un test
cursor.execute("DELETE FROM matches WHERE match_id = ?", (fake_id,))
conn.commit()
print("Après nettoyage :", match_already_collected(fake_id))  # attendu: False

---

**Checkpoint** : exécute ces cellules et confirme que le test de sanité affiche bien `False` / `True` / `False`. Ensuite on enchaîne sur le rate limiter proactif et la boucle de snowball sampling.

## Concept 3 — Rate limiter proactif

Riot impose deux limites simultanées pour une clé de dev : 20 requêtes/seconde **et** 100 requêtes/2 minutes. En usage soutenu (beaucoup d'appels d'affilée pendant la collecte), la limite des 2 minutes est de loin la plus contraignante (50 req/min en moyenne, très en dessous de 20/s) — donc respecter uniquement celle-ci suffit en pratique.

Le principe d'une **fenêtre glissante** : on garde en mémoire les instants (timestamps) des derniers appels. Avant chaque nouvel appel, on retire de cette mémoire les timestamps plus vieux que 120 secondes (ils ne comptent plus), puis on regarde combien il en reste. S'il y en a déjà 100, on attend le temps nécessaire pour que le plus ancien sorte de la fenêtre avant de continuer.

C'est différent de notre `riot_get` de la phase 1 : celui-ci attendait **après** avoir reçu un `429`. Ici, on évite complètement d'atteindre la limite.

In [ ]:
import time
from collections import deque


class RateLimiter:
    def __init__(self, max_calls, period_seconds):
        self.max_calls = max_calls
        self.period_seconds = period_seconds
        self.call_times = deque()  # timestamps des appels récents

    def wait_if_needed(self):
        now = time.time()

        # on oublie les appels trop vieux pour compter dans la fenêtre actuelle
        while self.call_times and now - self.call_times[0] > self.period_seconds:
            self.call_times.popleft()

        if len(self.call_times) >= self.max_calls:
            oldest_call = self.call_times[0]
            sleep_time = self.period_seconds - (now - oldest_call)
            if sleep_time > 0:
                print(f"Rate limiter : {sleep_time:.1f}s d'attente proactive...")
                time.sleep(sleep_time)

        self.call_times.append(time.time())

**Test du mécanisme** : avec de petites valeurs (3 appels max par tranche de 5 secondes), pour voir l'attente se déclencher en quelques secondes plutôt que d'attendre 2 minutes avec les vraies limites Riot.

In [ ]:
test_limiter = RateLimiter(max_calls=3, period_seconds=5)

for i in range(5):
    test_limiter.wait_if_needed()
    print(f"Appel {i + 1} effectué à t={time.time():.1f}")

**Ce que tu devrais observer** : les appels 1, 2, 3 s'enchaînent quasi instantanément (moins de 3 appels dans la fenêtre de 5s, pas d'attente). Au 4ème appel, le message `Rate limiter : ...s d'attente proactive...` apparaît, et l'appel est retardé jusqu'à ce que le 1er appel sorte de la fenêtre de 5 secondes.

## Concept 4 — Brancher le rate limiter sur `riot_get`

Ce notebook est indépendant de `01_riot_api_basics.ipynb` (kernel séparé) — on redéfinit donc rapidement le chargement de la clé API et les headers, puis on réécrit `riot_get` pour qu'il utilise le vrai `RateLimiter` (100 appels/120s) **avant** chaque requête, en plus de la gestion du `429` en filet de sécurité.

In [ ]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()
api_key = os.environ["RIOT_API_KEY"]
headers = {"X-Riot-Token": api_key}

region = "europe"  # routage régional, utilisé par Account-V1 et Match-V5

riot_limiter = RateLimiter(max_calls=100, period_seconds=120)


def riot_get(url, params=None, max_retries=3):
    for attempt in range(max_retries):
        riot_limiter.wait_if_needed()  # attente proactive, avant l'appel
        response = requests.get(url, headers=headers, params=params)
        if response.status_code == 429:
            retry_after = int(response.headers.get("Retry-After", 1))
            print(f"429 reçu malgré le rate limiter proactif, attente de {retry_after}s...")
            time.sleep(retry_after)
            continue
        return response
    raise RuntimeError("Trop de tentatives après rate limiting répété")

---

**Checkpoint** : exécute ces nouvelles cellules (le test du rate limiter, puis la redéfinition de `riot_get`). Il n'y a pas encore de vrai appel à l'API Riot dans ce concept — on vérifie juste que tout s'exécute sans erreur. Ensuite, dernière pièce du puzzle de la phase 2 : la boucle de crawl (snowball sampling) qui va tout assembler — `riot_get`, `RateLimiter`, et les fonctions SQLite du concept 2.

## Concept 5 — La boucle de crawl (snowball sampling)

C'est un parcours en largeur (BFS) sur le graphe des joueurs :
1. On part d'un `puuid` de départ, placé dans une **file d'attente** (`queue`, notre `deque`).
2. Tant que la file n'est pas vide et qu'on n'a pas atteint le nombre de matchs voulu : on prend le prochain joueur de la file, on récupère sa liste de matchs récents.
3. Pour chaque match pas encore en base (`match_already_collected`) : on télécharge le détail, on le sauvegarde (`save_match`), et on ajoute ses 10 participants à la file — nouvelles graines, obtenues "gratuitement" sans appel API dédié.
4. On marque le joueur comme exploré (`mark_player_explored`) pour ne jamais retraiter sa liste de matchs deux fois — c'est le check qui garantit la correction (voir discussion juste avant). Le check fait avant d'ajouter à la file reste comme optimisation partielle, en plus de celui-ci.

In [ ]:
def crawl_matches(seed_puuid, target_count, matches_per_player=10):
    queue = deque([seed_puuid])
    collected = 0

    while queue and collected < target_count:
        puuid = queue.popleft()

        if player_already_explored(puuid):
            continue
        mark_player_explored(puuid)

        match_ids_url = f"https://{region}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids"
        response = riot_get(match_ids_url, params={"start": 0, "count": matches_per_player})
        if response.status_code != 200:
            print(f"Erreur {response.status_code} pour le joueur {puuid}, on passe.")
            continue
        match_ids = response.json()

        for match_id in match_ids:
            if collected >= target_count:
                break
            if match_already_collected(match_id):
                continue

            match_detail_url = f"https://{region}.api.riotgames.com/lol/match/v5/matches/{match_id}"
            response = riot_get(match_detail_url)
            if response.status_code != 200:
                print(f"Erreur {response.status_code} pour le match {match_id}, on passe.")
                continue

            match_data = response.json()
            save_match(match_id, region, match_data)
            collected += 1

            # snowball : les 10 participants de ce match deviennent de nouvelles graines
            for participant_puuid in match_data["metadata"]["participants"]:
                if not player_already_explored(participant_puuid):
                    queue.append(participant_puuid)

            if collected % 10 == 0:
                print(f"{collected}/{target_count} matchs collectés...")

    print(f"Terminé : {collected} matchs collectés, {len(queue)} joueurs encore en attente dans la file.")

Petite précision sur `match_data["metadata"]["participants"]` : contrairement à `info.participants` (utilisé en phase 1 pour extraire champion/équipe/rôle), `metadata.participants` est directement une simple liste des 10 `puuid` de la partie — exactement ce qu'il nous faut ici pour alimenter la file, sans avoir à fouiller dans les objets détaillés de `info`.

## Récupérer le `puuid` de départ

Ce notebook est indépendant du premier — on refait rapidement l'appel Account-V1 (déjà expliqué en phase 1) pour obtenir la graine de départ du crawl.

In [ ]:
from urllib.parse import quote

game_name = "Spear Shot"
tag_line = "1111"

account_url = f"https://{region}.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{quote(game_name)}/{quote(tag_line)}"
response = riot_get(account_url)

seed_puuid = response.json()["puuid"]
print("Seed puuid:", seed_puuid)

## Premier test à petite échelle

Avant de viser des milliers de matchs (ce qui prendra du temps, vu le rate limiting), on lance une collecte de seulement **20 matchs** pour valider que tout le pipeline fonctionne correctement de bout en bout.

In [ ]:
crawl_matches(seed_puuid, target_count=20)

In [ ]:
cursor.execute("SELECT COUNT(*) FROM matches")
print("Total matchs en base:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM explored_players")
print("Total joueurs explorés:", cursor.fetchone()[0])

---

**Checkpoint final de la phase 2** : exécute tout, du seed `puuid` jusqu'à la vérification. Tu devrais voir `Total matchs en base: 20` et un nombre de joueurs explorés généralement plus petit que 20 (plusieurs matchs peuvent partager des joueurs déjà explorés).

Une fois validé, on aura le pipeline complet et testé. Pour la vraie collecte à grande échelle (plusieurs milliers de matchs), il suffira de relancer `crawl_matches` avec un `target_count` bien plus grand — sachant que ça prendra un certain temps (rate limiting) et qu'il faudra probablement renouveler la clé API Riot en cours de route (expiration 24h). On en discutera quand tu voudras lancer la vraie collecte.

## Bonus — Visualiser le contenu de la base avec pandas

`pandas` sait exécuter une requête SQL directement sur une connexion `sqlite3` et renvoyer le résultat sous forme de tableau (`DataFrame`), avec un affichage propre dans Jupyter — pratique pour explorer visuellement sans écrire de boucle Python à la main.

In [ ]:
import pandas as pd

df = pd.read_sql_query("SELECT match_id, region FROM matches", conn)
df

On n'affiche pas `raw_json` ici : c'est un gros bloc de texte, illisible dans un tableau. On ira le décortiquer (`json.loads` + extraction champion/équipe/rôle) en phase 3, quand on construira le vrai dataset d'entraînement.